# 03 · Calibração

**Bloco 3 do workshop.**

Pergunta: *quando o modelo diz 0,87, é 0,87 mesmo?*

Um modelo é calibrado se, entre todos os casos em que ele disse 0,70, cerca de 70% forem realmente positivos. Ordenar bem e estimar probabilidade bem são propriedades independentes.

In [ ]:
import numpy as np
from sklearn.datasets import make_classification
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier

# dataset canônico do workshop — NÃO altere estes parâmetros
X, y = make_classification(n_samples=20000, n_features=20, n_informative=8,
                           n_redundant=4, weights=[0.99, 0.01], flip_y=0.0,
                           class_sep=1.5, random_state=42)

X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.3,
                                          stratify=y, random_state=42)
# split extra para calibrar no notebook 03 (nunca calibre no teste)
X_tr2, X_val, y_tr2, y_val = train_test_split(X_tr, y_tr, test_size=0.25,
                                              stratify=y_tr, random_state=42)

clf = RandomForestClassifier(n_estimators=200, random_state=42, n_jobs=-1).fit(X_tr, y_tr)
proba = clf.predict_proba(X_te)[:, 1]

print(f"teste: {len(y_te)} casos | {int(y_te.sum())} fraudes | prevalência {y_te.mean():.2%}")

## 3.1 Modelo treinado sem o conjunto de calibração

**Regra que não tem exceção:** calibre em um conjunto separado. Calibrar no teste e reportar no mesmo teste é vazamento.

In [ ]:
clf2  = RandomForestClassifier(n_estimators=200, random_state=42, n_jobs=-1).fit(X_tr2, y_tr2)
p_sem = clf2.predict_proba(X_te)[:, 1]
print("treino:", len(y_tr2), "| calibração:", len(y_val), "| teste:", len(y_te))

## 3.2 Diagrama de confiabilidade

In [ ]:
import matplotlib.pyplot as plt
from sklearn.calibration import calibration_curve

pt, pp = calibration_curve(y_te, p_sem, n_bins=8, strategy='quantile')

plt.figure(figsize=(5.5, 5.5))
plt.plot(pp, pt, 'o-', label='modelo')
plt.plot([0, max(pp)], [0, max(pp)], '--', color='gray', label='perfeitamente calibrado')
plt.xlabel('probabilidade prevista'); plt.ylabel('frequência observada')
plt.legend(); plt.tight_layout(); plt.show()

Acima da diagonal, o modelo é **sub-confiante**. Abaixo, **super-confiante**.

## 3.3 ECE e Brier

In [ ]:
from sklearn.metrics import brier_score_loss

def ece(y_true, p, n_bins=10):
    bins = np.linspace(0, 1, n_bins+1)
    idx  = np.clip(np.digitize(p, bins[1:-1]), 0, n_bins-1)
    e = 0.0
    for b in range(n_bins):
        m = idx == b
        if m.sum() == 0:
            continue
        e += m.mean() * abs(y_true[m].mean() - p[m].mean())
    return e

print(f"ECE:   {ece(y_te, p_sem):.4f}   (compare com a prevalência: {y_te.mean():.4f})")
print(f"Brier: {brier_score_loss(y_te, p_sem):.5f}")

## 3.4 Correções

In [ ]:
from sklearn.calibration import CalibratedClassifierCV
from sklearn.metrics import roc_auc_score

# scikit-learn >= 1.6
from sklearn.frozen import FrozenEstimator
iso   = CalibratedClassifierCV(FrozenEstimator(clf2), method='isotonic').fit(X_val, y_val)
platt = CalibratedClassifierCV(FrozenEstimator(clf2), method='sigmoid').fit(X_val, y_val)
# em versões < 1.6:
# iso = CalibratedClassifierCV(clf2, method='isotonic', cv='prefit').fit(X_val, y_val)

p_iso   = iso.predict_proba(X_te)[:, 1]
p_platt = platt.predict_proba(X_te)[:, 1]

print(f"{'':<14}{'ECE':>9}{'Brier':>11}{'AUC':>9}")
for nome, p in [("sem calibrar", p_sem), ("Platt", p_platt), ("Isotonic", p_iso)]:
    print(f"{nome:<14}{ece(y_te, p):>9.4f}{brier_score_loss(y_te, p):>11.5f}{roc_auc_score(y_te, p):>9.4f}")

### ✏️ Tarefa — leia a tabela acima

1. O modelo era super-confiante, sub-confiante ou calibrado?
2. A AUC mudou com Platt? E com isotonic? **Por quê?**
3. Depois de calibrar, o limiar ótimo do notebook 02 ainda vale? (Pense antes de responder — a calibração desloca a escala.)

In [ ]:
resposta = """
1.
2.
3.
"""
print(resposta)